In [ ]:
import os
import sys
import numpy as np

from SciExpeM_API.SciExpeM import SciExpeM
from SciExpeM_API.Models import *

sciexp_scripts_path = r"/Users/lpratalimaffei/Library/CloudStorage/OneDrive-PolitecnicodiMilano/Luna/Universita/RICERCA/SCIEXPEM/SciExpeM_API/Examples/scripts"
sys.path.append(sciexp_scripts_path)

import extract_data
import build_sciexp_objects

my_sciexpem = SciExpeM(username='YOUR_USERNAME',
                       password='YOUR_PASSWORD', secure=False, port=8080)

CONVERT_TO_BAR = {'atm': 1.01325, 'bar': 1., 'Torr': 0.00133322, 'Pa': 1e-5}


### FilePaper
 - description*
 - reference_doi*
 - author*
 - title*
 - year*
 - volume
 - page
 - journal

In [2]:
#TEMPLATE

file_paper = FilePaper(reference_doi="10.1021/jp982177+",
                       title="Pyrolysis And Oxidation Of Phenol",
                       author="Brezinsky, K.; Pecullan, M.; Glassman, I.",
                       year=1998,
                       description="Brezinsky, K.; Pecullan, M.; Glassman, I. - The Journal Of Physical Chemistry A, 1998-10-01, (102), 8614-8619"
                       )

### OpenSMOKE input file if available

In [3]:
datapath = os.path.join(sciexp_scripts_path, 'data')
osinputname = 'Flowtime_inputOS.dic'
# WARNING LIST OF SPECIES MUST HAVE THE NAMES OF THE MECH YOU ARE GOING TO SIMULATE - OTHERWISE, NEED REPLACEMENT
inputstr, extrainfo = extract_data.process_osinput(datapath, osinputname, profiles=False, flameinfo=True)  
# LA LISTA DI OUTPUTSPECIES DEVE AVERE SPECIE NEL MECCANISMO CHE SIMULERAI

In [4]:
print(extrainfo)

{'profileinfo': {}, 'commonprop': {'residence time': ['1', 's'], 'temperature': ['1169.', 'K'], 'pressure': ['1', 'atm']}, 'character': {}, 'molefractions': [['C6H5OH', 'O2', 'N2'], ['0.000533', '0.005830', '0.993637']], 'oxidizermolefractions': []}


### Common properties

- name
- units
- value
- source_type

In [5]:
sourcetype = 'reported'
commonprop = []
for name, values in extrainfo['commonprop'].items():
    print(name, values)
    if name not in ['residence time', 'temperature']: #those are not common props for flow reacs that are non-isothermal
        ci = CommonProperty(name=name, units=values[1], value=values[0], source_type=sourcetype)
        commonprop.append(ci)
    
# ADD OTHER COMMON PROPERTIES
#ci = CommonProperty(name='temperature', units='K', value='1120', source_type='reported')
#commonprop.append(ci)
############# DO NOT EDIT
# extract pressure
if 'pressure' in extrainfo['commonprop'].keys():
    Pval, Punit = extrainfo['commonprop']['pressure']
    P = float(Pval) * CONVERT_TO_BAR[Punit]


residence time ['1', 's']
temperature ['1169.', 'K']
pressure ['1', 'atm']


In [6]:
for c in commonprop:
    print(c.serialize())

{'name': 'pressure', 'units': 'atm', 'value': '1', 'source_type': 'reported'}


### Initial Species

- name
- units
- value
- source_type
- configuration

In [7]:
comp_unit = 'mole fraction'
srctype = 'reported'
config = 'premixed'
# do not edit
inspecies, fuelobjs = build_sciexp_objects.build_initial_species(
    my_sciexpem=my_sciexpem,
    molefractions=extrainfo['molefractions'],
    source_type=srctype,
    units=comp_unit,
    configuration=config,
)

Species: ['C6H5OH', 'O2', 'N2']
Fuels: ['C6H5OH']
Mole Frac: ['0.000533', '0.005830', '0.993637']
config: ['premixed', 'premixed', 'premixed']


In [9]:
sp_table = {}

In [10]:
# read data
datafile = os.path.join(sciexp_scripts_path, 'data', 'Flowtime_data.txt')
df_data = extract_data.readdata(datafile, delzero=True)
# process data
datacols = []
srctype = 'calculated'
label = 'experimental_data'
#x = ['temperature', 'K']
x = ['time', 's'] # NB must be 'residence time' for concentration time profile, but only works with 'time' lol
#y = ['concentration', 'mol/cm3']
# x = ['temperature', 'K']
y = ['composition', 'mole fraction']
#uncert_x = [30, 'absolute'] #currently unavailable
list_exclude_species = []
uncert_x = [] #currently unavailable
uncert_y = [] # uncomment above to put uncertainty
#################### DO NOT EDIT################
TINF, TSUP = extract_data.temperature_bounds(
    extrainfo,
    x=x,
    df_data=df_data,
)
print('T max and min: {} - {} K'.format(TINF, TSUP))
############## check residence time from data
commonprop = build_sciexp_objects.update_residence_time_from_data(commonprop, df_data, x)

########## build datacolumns
datacols = build_sciexp_objects.build_data_columns(
    df_data=df_data,
    my_sciexpem=my_sciexpem,
    x=x,
    y=y,
    source_type=srctype,
    label=label,
    sp_table=sp_table,
    list_exclude_species=list_exclude_species,
    uncert_x=uncert_x,
    uncert_y=uncert_y,
)

T max and min: 1169.0 - 1169.0 K
C6H5OH [<Species (49)>]
CO [<Species (4)>]
CO2 [<Species (10)>]
C2H2 [<Species (51)>]
C4H6 [<Species (62)>]
C5H6 [<Species (64)>]
C6H6 [<Species (39)>]
C10H8 [<Species (27)>]
species C3H4 not as preferred name: alternative names searched
C3H4 [<Species (15)>, <Species (93)>]
CH4 [<Species (6)>]


### Assemble the experiment

In [11]:
PHI_INF = extract_data.equivalence_ratio(extrainfo['molefractions'])
PHI_SUP = PHI_INF

e = Experiment(reactor='flow reactor',  # [flow reactor, flame, stirred reactor ]
               experiment_type='concentration time profile measurement',
               file_paper=file_paper,
               reactor_modes=['premixed'], # da mettere in tutte le fiamme # coflow, couterflow, premixed, burner stabilized stagnation
               data_columns=datacols, 
               initial_species=inspecies, 
               common_properties=commonprop,
               os_input_file=inputstr,
               t_inf = TINF, t_sup = TSUP,
              # t_inf = TINF, t_sup = TSUP,
               p_inf = P, p_sup = P,
               phi_inf = PHI_INF, phi_sup = PHI_SUP,
               # fuels=fuels,
               fuels_object = fuelobjs, # devi passargli gli id delle specie che sono fuels
               comment = 'simul should be shifted'
               #comment = 'technically not isothermal but simulated as such - T almost unvaried'
               #comment = 'C4H2 plot unclear, order might be 1e6 or 1e8 (person who digitized assumed 1e8); bath gas varies with experimental run (Ne, He), but should not affect the results'
               )

In [ ]:
e.serialize()
# check that everything is ok

### Send Experiment

In [12]:
my_sciexpem.insertElement(e, verbose=True)

Experiment element inserted successfully.
